# 05 — Diagnostic
## The σ-face Faraday test: does a screen exist at the DM κ peaks?

**This is the measurement notebook.**

Rules:
- Threshold is read from `constants.py` — it was set before any data was examined
- The raw ΔRM value is the answer. There is no post-hoc adjustment.
- Failed or marginal results stay in the output, period.
- Both synthetic models are tested first (engine validation), then real data if available.

---

In [ ]:
import sys, os, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '../modules')
from constants import (RA0, DEC0, DM_NW, DM_SE, GAS_NW, GAS_SE,
                       RM_THRESHOLD_RADM2, POL_FRAC_BRIGHT_THRESHOLD,
                       POL_FRAC_DARK_THRESHOLD)
from transect import (transect_endpoints, sample_points,
                      extract_rm_from_qu_cubes, compute_diagnostic, print_diagnostic)

BASE   = Path('/media/rendier/0123-4567/bullet_cluster')
SYNTH  = BASE / 'radio/meerkat/synthetic'
OUT    = Path('../output')
OUT.mkdir(exist_ok=True)

print(f'DIAGNOSTIC THRESHOLD: |ΔRM| > {RM_THRESHOLD_RADM2} rad/m² (set in constants.py)')
print(f'Pol fraction bright:  p > {POL_FRAC_BRIGHT_THRESHOLD} (wave DM: lensing amplification)')
print(f'Pol fraction dark:    p < {POL_FRAC_DARK_THRESHOLD} (particle DM: beam depolarisation)')
print()
print('These values do not change after this cell is executed.')

In [ ]:
# --- Engine validation: must recover correct answer from synthetic ---
T   = transect_endpoints(length_arcmin=10.0)
ra_arr, dec_arr, offset = sample_points(T, n_points=200)

validation_results = {}

for dm_model in ['wave', 'particle']:
    q = SYNTH / f'synthetic_Q_{dm_model}.fits'
    u = SYNTH / f'synthetic_U_{dm_model}.fits'
    f = SYNTH / f'synthetic_freqs_{dm_model}.dat'
    if not q.exists():
        print(f'Synthetic {dm_model} cubes missing — run notebook 03')
        continue

    rm, pi, evpa = extract_rm_from_qu_cubes(str(q), str(u), str(f), ra_arr, dec_arr)

    # Kappa: use viewer PNG if available, else analytical NFW on transect
    from PIL import Image
    from transect import _interp2d
    kappa_png = BASE / 'viewer/layers/lensing_kappa.png'
    if kappa_png.exists():
        SIZE, SCALE = 800, 1.0/3600
        img  = np.array(Image.open(kappa_png).convert('L')).astype(float)
        px   = SIZE/2 - (ra_arr - RA0) / SCALE
        py   = SIZE/2 + (dec_arr - DEC0) / SCALE
        kappa = _interp2d(img, px, py)
    else:
        # Fallback: toy two-Gaussian κ model
        kappa = np.zeros_like(ra_arr)
        for dm in [DM_NW, DM_SE]:
            dr = (ra_arr - dm['ra']) * np.cos(np.radians(dm['dec'])) * 60
            dd = (dec_arr - dm['dec']) * 60
            kappa += 0.3 * np.exp(-(dr**2 + dd**2) / (1.5**2))

    diag = compute_diagnostic(rm_profile=rm, kappa_profile=kappa, offset_arcmin=offset)
    validation_results[dm_model] = diag

    expected = 'wave' if dm_model == 'wave' else 'particle'
    got      = diag.get('dm_model_favoured', 'unknown')
    ok       = '✓ PASS' if got == expected else '✗ FAIL'

    print(f'\n{"─"*50}')
    print(f'Synthetic model: {dm_model}   Expected: {expected}')
    print_diagnostic(diag)
    print(f'Engine validation: {ok}')

In [ ]:
# --- Visualise the diagnostic: RM profile with ΔRM annotations ---
fig, axes = plt.subplots(1, len(validation_results), figsize=(7*len(validation_results), 5),
                         sharey=True)
if len(validation_results) == 1:
    axes = [axes]

for ax, (dm_model, diag) in zip(axes, validation_results.items()):
    rm = diag.get('rm_profile')
    if rm is None: continue
    ax.plot(offset, rm, 'k', lw=0.8, label='RM(λ²)')

    # Shade ΔRM at each κ peak
    for d in diag.get('delta_rm_at_dm_peaks', []):
        x  = d['offset']
        rm_m = d['rm_measured']
        rm_b = d['rm_baseline']
        col = 'red' if abs(d['delta_rm']) > RM_THRESHOLD_RADM2 else 'green'
        ax.annotate('', xy=(x, rm_m), xytext=(x, rm_b),
                    arrowprops=dict(arrowstyle='<->', color=col, lw=2))
        ax.text(x + 0.15, (rm_m + rm_b)/2,
                f'ΔRM={d["delta_rm"]:+.1f}', color=col, fontsize=8)

    # Mark DM positions
    for dm, ls in [(DM_NW, '--'), (DM_SE, ':')]:
        dr = (dm['ra']  - T['mid_ra'])  * np.cos(np.radians(T['mid_dec'])) * 60
        dd = (dm['dec'] - T['mid_dec']) * 60
        x  = np.sqrt(dr**2 + dd**2) * np.sign(dr + dd)
        ax.axvline(x, color='blue', ls=ls, lw=0.8, label=dm['label'])

    ax.axhline(0, color='grey', lw=0.5)
    ax.set_xlabel('Transect offset (arcmin)')
    ax.set_ylabel('RM (rad/m²)')
    verdict = diag.get('verdict', '—')
    ax.set_title(f'Synthetic {dm_model} DM\n{verdict[:50]}')
    ax.legend(fontsize=7)

plt.suptitle('ΔRM diagnostic — arrows show excess at κ peaks')
plt.tight_layout()
plt.savefig(OUT / 'diagnostic_rm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/diagnostic_rm_comparison.png')

In [ ]:
# --- Real data diagnostic (runs if MGCLS cubes are present) ---
real_q = BASE / 'radio/meerkat/MGCLS_1E0657-558_Q_cube.fits'
real_u = BASE / 'radio/meerkat/MGCLS_1E0657-558_U_cube.fits'
real_f = BASE / 'radio/meerkat/MGCLS_freqs.dat'

if real_q.exists() and real_u.exists():
    print('REAL MGCLS DATA FOUND — running diagnostic on real Stokes cubes...')
    rm_r, pi_r, evpa_r = extract_rm_from_qu_cubes(
        str(real_q), str(real_u), str(real_f), ra_arr, dec_arr
    )
    diag_real = compute_diagnostic(rm_profile=rm_r, kappa_profile=kappa,
                                   offset_arcmin=offset)
    print_diagnostic(diag_real)

    # Save
    with open(OUT / 'diagnostic_real_data.json', 'w') as fp:
        summary = {
            'verdict':           diag_real.get('verdict'),
            'dm_model_favoured': diag_real.get('dm_model_favoured'),
            'max_abs_delta_rm':  float(diag_real.get('max_abs_delta_rm', np.nan)),
            'threshold_radm2':   RM_THRESHOLD_RADM2,
            'data_source':       'REAL MGCLS SARAO SSV-20180624-FC-01',
            'delta_rm_detail':   diag_real.get('delta_rm_at_dm_peaks', []),
        }
        json.dump(summary, fp, indent=2, default=float)
    print('Saved: output/diagnostic_real_data.json')
else:
    print('Real MGCLS cubes not yet available.')
    print('Register at: https://archive.sarao.ac.za/')
    print('Proposal: SSV-20180624-FC-01')
    print('Request: Stokes Q and U enhanced image cubes for 1E 0657-558')
    print()
    print('When cubes arrive, drop them in:')
    print(f'  {real_q}')
    print(f'  {real_u}')
    print('Then re-run this cell.')